# List-aware · single-fit fixed-origin follow-up

**Goal:** compare the full-strength CatBoost MAE, CatBoost Huber, CatBoost spatial/advanced, XGBoost spatial, LightGBM spatial, and predefined geometric blends over August–October 2025, **without refitting between months**.

Runs `listaware_fixed_origin_followup.py` (6 model fits total rather than 6 fits per month). The final predeclared blends include the leading 50/35/15 blend combined with CatBoost MAE at 80/20, spatial/advanced CatBoost 70/30, and Huber/MAE 70/30.

All months are previously inspected historical development data, not untouched confirmation. **The last cell can also recompute new blends from your existing expanding-run CSV without training.**

In [6]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "listaware_fixed_origin_followup.py").exists():
    raise FileNotFoundError("Open this root notebook from the repository root.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from listaware_fixed_origin_followup import run_followup, ADDITIONAL_BLENDS
print("Root:", ROOT)
display(pd.DataFrame([
    {"blend": name, "weights": ", ".join(f"{member} {w:.0%}" for member,w in specs.items())}
    for name, specs in ADDITIONAL_BLENDS.items()
]))

Root: C:\Users\deant\Desktop\Coding\Data Dash Case Comp


,blend,weights
0,followup_winner_plus_mae,"cat_huber_spline_geo 40%, xgb_spline_geo 28%, ..."
1,followup_two_cat_spatial_advanced,"cat_huber_spline_geo 70%, cat_huber_advanced 30%"
2,followup_cat_mae_xgb_lgb,"cat_mae 50%, xgb_spline_geo 35%, lgb_spline_ge..."
3,followup_huber_mae,"cat_huber_baseline 70%, cat_mae 30%"
4,followup_cat_spatial_mae,"cat_huber_spline_geo 70%, cat_mae 30%"


## 1 · Run a short fixed-origin pilot

This pilot uses the same prior-sales cutoff (August 1, 2025), capped historical training rows and boosting rounds. It **does not** replace the full-fidelity confirmation.

In [8]:
RUN_PILOT = True         # Set True to run / resume; quick preflight recommended.
PILOT_OUTPUT = Path("outputs/listaware_fixed_origin_pilot")
if RUN_PILOT:
    pilot_pooled, pilot_monthly, pilot_types, pilot_preds = run_followup(
        full=False, output_dir=PILOT_OUTPUT, threads=4, bootstrap_repeats=300
    )
    display(pilot_pooled.round(4))
else:
    print("Pilot disabled; set RUN_PILOT=True to fit six small historical models.")

Fixed-origin cat_huber_baseline: ONE fit on 10,000 earlier sales; predict 1,050 future homes
Fixed-origin cat_mae: ONE fit on 10,000 earlier sales; predict 1,050 future homes
Fixed-origin cat_huber_spline_geo: ONE fit on 10,000 earlier sales; predict 1,050 future homes
Fixed-origin cat_huber_advanced: ONE fit on 10,000 earlier sales; predict 1,050 future homes
Fixed-origin xgb_spline_geo: ONE fit on 10,000 earlier sales; predict 1,050 future homes
Fixed-origin lgb_spline_geo: ONE fit on 10,000 earlier sales; predict 1,050 future homes

List-aware fixed-origin / existing-OOT blend scoreboard:
  arm                           variant  homes    mdape     mape   p90_ape  delta_mdape_pp  delta_mape_pp
model                cat_huber_baseline   1050 4.497423 5.678073 11.599750        0.000000       0.000000
blend          followup_cat_spatial_mae   1050 4.536998 5.620516 11.593920        0.039575      -0.057557
model              cat_huber_spline_geo   1050 4.537167 5.618990 11.674916        0

,homes,mape,mdape,median_signed,arm,variant,delta_mdape_pp,delta_mape_pp,p90_ape,homes_improved_fraction,delta_mdape_ci_low,delta_mdape_ci_high,delta_mape_ci_low,delta_mape_ci_high
0,1050,5.6781,4.4974,1.1893,model,cat_huber_baseline,0.0000,0.0000,11.5997,0.0000,0.0000,0.0000,0.0000,0.0000
13,1050,5.6205,4.5370,1.0683,blend,followup_cat_spatial_mae,0.0396,-0.0576,11.5939,0.5276,-0.1642,0.1489,-0.0930,-0.0168
2,1050,5.6190,4.5372,0.8929,model,cat_huber_spline_geo,0.0397,-0.0591,11.6749,0.5229,-0.2025,0.1585,-0.1047,-0.0105
12,1050,5.6651,4.5478,1.1957,blend,followup_huber_mae,0.0504,-0.0130,11.5438,0.5190,-0.0930,0.1227,-0.0259,-0.0009
1,1050,5.6499,4.5586,1.1669,model,cat_mae,0.0611,-0.0282,11.4116,0.5133,-0.1582,0.1993,-0.0695,0.0148
3,1050,5.6239,4.5596,1.0394,model,cat_huber_advanced,0.0622,-0.0541,11.6258,0.5190,-0.1504,0.2205,-0.1013,-0.0106
10,1050,5.6167,4.5683,0.8728,blend,followup_two_cat_spatial_advanced,0.0708,-0.0613,11.5946,0.5267,-0.1805,0.1804,-0.1036,-0.0168
8,1050,5.6266,4.6060,1.3634,blend,blend_cat_baseline_spatial_xgb,0.1086,-0.0514,11.4897,0.5457,-0.1314,0.2734,-0.0944,-0.0061
11,1050,5.6407,4.6726,1.4720,blend,followup_cat_mae_xgb_lgb,0.1752,-0.0374,11.2942,0.5267,-0.1667,0.3627,-0.1138,0.0449
6,1050,5.6159,4.6735,1.2554,blend,blend_cat_xgb_spatial,0.1761,-0.0622,11.4276,0.5210,-0.1305,0.3260,-0.1211,0.0018


## 2 · Full-strength fixed-origin test

This evaluates all historical audit homes, fits each model **once** on all prior eligible training sales, and predicts all three audit months without updating the models. Predictions and blended score tables resume from `outputs/listaware_fixed_origin_full/`.

**Choose `RUN_FULL=True` explicitly**; six trained models with full iterations still require nontrivial compute.

In [9]:
RUN_FULL = True
FULL_OUTPUT = Path("outputs/listaware_fixed_origin_full")
if RUN_FULL:
    pooled, monthly, by_type, preds = run_followup(
        full=True, output_dir=FULL_OUTPUT, threads=4,
        bootstrap_repeats=500
    )
else:
    folder = FULL_OUTPUT / "fixed_weight_followup"
    if (folder / "followup_pooled.csv").is_file():
        pooled = pd.read_csv(folder / "followup_pooled.csv")
        monthly = pd.read_csv(folder / "followup_monthly.csv")
        print("Loaded existing fixed-origin scores.")
    else:
        pooled = monthly = None
        print("Full run disabled. Set RUN_FULL=True when ready.")
if pooled is not None:
    display(pooled[[
        "arm","variant","homes","mdape","mape","p90_ape",
        "delta_mdape_pp","delta_mape_pp","delta_mdape_ci_low",
        "delta_mdape_ci_high",
    ]].round(4))

Fixed-origin cat_huber_baseline: ONE fit on 132,113 earlier sales; predict 5,951 future homes
Fixed-origin cat_mae: ONE fit on 132,113 earlier sales; predict 5,951 future homes
Fixed-origin cat_huber_spline_geo: ONE fit on 132,113 earlier sales; predict 5,951 future homes
Fixed-origin cat_huber_advanced: ONE fit on 132,113 earlier sales; predict 5,951 future homes
Fixed-origin xgb_spline_geo: ONE fit on 132,113 earlier sales; predict 5,951 future homes
Fixed-origin lgb_spline_geo: ONE fit on 132,113 earlier sales; predict 5,951 future homes

List-aware fixed-origin / existing-OOT blend scoreboard:
  arm                           variant  homes    mdape     mape   p90_ape  delta_mdape_pp  delta_mape_pp
blend          followup_cat_mae_xgb_lgb   5951 3.899564 4.853451 10.040185       -0.082719      -0.092176
model                           cat_mae   5951 3.923723 4.936018 10.294528       -0.058559      -0.009609
blend                 blend_cat_xgb_lgb   5951 3.926443 4.865653 10.101903   

,arm,variant,homes,mdape,mape,p90_ape,delta_mdape_pp,delta_mape_pp,delta_mdape_ci_low,delta_mdape_ci_high
11,blend,followup_cat_mae_xgb_lgb,5951,3.8996,4.8535,10.0402,-0.0827,-0.0922,-0.1473,-0.0120
1,model,cat_mae,5951,3.9237,4.9360,10.2945,-0.0586,-0.0096,-0.1102,-0.0034
7,blend,blend_cat_xgb_lgb,5951,3.9264,4.8657,10.1019,-0.0558,-0.0800,-0.1334,0.0317
9,blend,followup_winner_plus_mae,5951,3.9445,4.8588,10.0916,-0.0378,-0.0868,-0.1167,0.0168
8,blend,blend_cat_baseline_spatial_xgb,5951,3.9511,4.8823,10.1234,-0.0311,-0.0633,-0.0905,0.0221
13,blend,followup_cat_spatial_mae,5951,3.9660,4.9355,10.3226,-0.0163,-0.0101,-0.0665,0.0267
6,blend,blend_cat_xgb_spatial,5951,3.9692,4.8791,10.0891,-0.0131,-0.0665,-0.0715,0.0466
12,blend,followup_huber_mae,5951,3.9786,4.9369,10.2774,-0.0037,-0.0087,-0.0468,0.0136
0,model,cat_huber_baseline,5951,3.9823,4.9456,10.2941,0.0000,0.0000,0.0000,0.0000
2,model,cat_huber_spline_geo,5951,3.9914,4.9456,10.3038,0.0091,-0.0001,-0.0477,0.0519


In [4]:
import matplotlib.pyplot as plt
if monthly is not None:
    display(monthly.pivot_table(index="variant",columns="month",values="delta_mdape_pp").round(3))
if pooled is not None:
    plot = pooled.sort_values("delta_mdape_pp")
    fig,ax = plt.subplots(figsize=(10, max(4,.38*len(plot))))
    ax.barh(plot.variant,plot.delta_mdape_pp)
    ax.axvline(0,color="gray")
    ax.invert_yaxis()
    ax.set_xlabel("Change in MdAPE vs fixed-origin CatBoost Huber (percentage points)")
    plt.tight_layout()
    plt.show()

ModuleNotFoundError: No module named 'matplotlib.backends.registry'

## 3 · Zero-training blend analysis of your EXISTING full-strength expanding predictions

If you have already completed `outputs/listaware_full_shortlist/predictions.csv`, this cell calculates the **new predeclared blends immediately**. It doesn't retrain any model and it does not relabel the existing expanding forecasts as fixed-origin predictions.

This is useful for deciding which blend hypotheses merit the expensive fixed-origin check.

In [12]:
RUN_EXISTING_EXPANDING_ANALYSIS = True
EXPANDING_PREDICTIONS = Path("outputs/listaware_full_shortlist/predictions.csv")
if RUN_EXISTING_EXPANDING_ANALYSIS and EXPANDING_PREDICTIONS.is_file():
    expanding_pooled, expanding_monthly, _, _ = run_followup(
        from_predictions=EXPANDING_PREDICTIONS,
        output_dir=Path("outputs/listaware_existing_expanding_blend_followup"),
        bootstrap_repeats=500,
    )
    display(expanding_pooled[["arm","variant","homes","mdape","mape","delta_mdape_pp"]].round(4))
else:
    print("No expanding CSV found; skip scores-only comparison.")

No expanding CSV found; skip scores-only comparison.


## Interpretation

Compare the original 50/35/15 blend, CatBoost MAE and the 80/20 blend on exactly the same homes; look for month-to-month consistency. Prefer **fixed-origin** for the harder forward-horizon check. Results from these previously inspected months remain exploratory, even if a paired bootstrap interval excludes zero.